# Notebook 2: Variant Quality Control - SOLUTIONS

**This is the instructor version with complete solutions.**

**Do NOT distribute to students before the exercise.**

In [ ]:
import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt

scripts_dir = "/workspace/scripts"
if not os.path.exists(scripts_dir):
    scripts_dir = os.path.join(os.path.dirname(os.getcwd()), "scripts")
sys.path.insert(0, scripts_dir)

from timing_utils import time_operation, compare_operations, partition_info
from viz_utils import plot_chromosome_distribution, plot_quality_histogram, plot_af_distribution

spark = SparkSession.builder \
    .appName("BINFX410_Variant_QC_Solutions") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

data_dir = "/workspace/data"
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")

print(f"Spark UI: http://localhost:4040")
print("Setup complete!")

---
## Task 1: Load and Explore Data - SOLUTION

In [ ]:
# Load the medium variant dataset
variants = spark.read.csv(
    os.path.join(data_dir, "variants_medium.csv"),
    header=True,
    inferSchema=True
)

variants.printSchema()
variants.show(10, truncate=False)

total = variants.count()
print(f"\nTotal variants: {total:,}")

In [ ]:
# Summary statistics
variants.select("QUAL", "DP", "AF", "GQ").summary().show()

n_chroms = variants.select("CHROM").distinct().count()
print(f"Distinct chromosomes: {n_chroms}")

# FILTER and CONSEQUENCE distributions
variants.groupBy("FILTER").count().orderBy("count", ascending=False).show()
variants.groupBy("CONSEQUENCE").count().orderBy("count", ascending=False).show()

---
## Task 2: Basic Quality Filtering - SOLUTION

In [ ]:
# Apply GATK hard filters
filtered_variants = variants.filter(
    (F.col("QUAL") >= 30) &
    (F.col("DP") >= 10) &
    (F.col("GQ") >= 20)
)

total = variants.count()
passed = filtered_variants.count()

print(f"Total variants: {total:,}")
print(f"Passed filters: {passed:,}")
print(f"Filtered out: {total - passed:,} ({(total - passed) / total * 100:.1f}%)")

In [ ]:
# Individual filter analysis
qual_pass = variants.filter(F.col("QUAL") >= 30).count()
dp_pass = variants.filter(F.col("DP") >= 10).count()
gq_pass = variants.filter(F.col("GQ") >= 20).count()

print(f"Pass QUAL >= 30: {qual_pass:,} ({qual_pass/total*100:.1f}%)")
print(f"Pass DP >= 10:   {dp_pass:,} ({dp_pass/total*100:.1f}%)")
print(f"Pass GQ >= 20:   {gq_pass:,} ({gq_pass/total*100:.1f}%)")

# ANSWER: Filter analysis depends on data generation, but typically QUAL
# removes the most because of the bimodal distribution (~20% low quality).
# Filtering is a narrow transformation - NO shuffle required.

---
## Task 3: Per-Chromosome Statistics - SOLUTION

In [ ]:
chrom_stats = variants.groupBy("CHROM").agg(
    F.count("*").alias("variant_count"),
    F.round(F.mean("QUAL"), 2).alias("mean_qual"),
    F.round(F.mean("DP"), 2).alias("mean_depth"),
    F.sum(F.when(F.col("FILTER") == "PASS", 1).otherwise(0)).alias("pass_count")
).withColumn(
    "pass_rate", F.round(F.col("pass_count") / F.col("variant_count"), 3)
)

# Sort chromosomes properly (chr1, chr2, ..., chr22, chrX, chrY)
chrom_stats_sorted = chrom_stats.withColumn(
    "chrom_sort",
    F.when(F.col("CHROM") == "chrX", F.lit("23"))
     .when(F.col("CHROM") == "chrY", F.lit("24"))
     .otherwise(F.lpad(F.regexp_extract("CHROM", r"chr(\d+)", 1), 2, "0"))
).orderBy("chrom_sort").drop("chrom_sort")

chrom_stats_sorted.show(24, truncate=False)

In [ ]:
# Visualization
chrom_stats_pd = chrom_stats.toPandas()
fig = plot_chromosome_distribution(chrom_stats_pd, "variant_count",
                                    title="Variant Count by Chromosome")
plt.show()

# ANSWER: chr1 has the highest variant count because it is the longest
# chromosome. The groupBy has 2 stages: partial aggregation within each
# partition, then final aggregation after shuffling by CHROM key.

---
## Task 4: AF Stratification - SOLUTION

In [ ]:
variants_classified = variants.withColumn(
    "frequency_class",
    F.when(F.col("AF") < 0.001, "Singleton")
     .when(F.col("AF") < 0.01, "Rare")
     .when(F.col("AF") < 0.05, "Low frequency")
     .otherwise("Common")
)

variants_classified.groupBy("frequency_class") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [ ]:
# Cross-tabulation using pivot
freq_by_chrom = variants_classified.groupBy("CHROM") \
    .pivot("frequency_class", ["Singleton", "Rare", "Low frequency", "Common"]) \
    .count() \
    .fillna(0)

freq_by_chrom.show(24, truncate=False)

# ANSWER: Rare variants are NOT uniformly distributed because:
# 1. Longer chromosomes have more variants overall
# 2. Some regions are under stronger selection
# 3. Mutation rate varies across the genome

---
## Task 5: Gene Annotation Join - SOLUTION

In [ ]:
genes = spark.read.csv(
    os.path.join(data_dir, "genes.bed"),
    sep="\t", header=False,
    schema="chrom STRING, start INT, end INT, gene_name STRING, strand STRING"
)
print(f"Loaded {genes.count()} gene annotations")
genes.show(5)

In [ ]:
# Positional join
variant_genes = variants.join(
    genes,
    (variants.CHROM == genes.chrom) &
    (variants.POS >= genes.start) &
    (variants.POS <= genes.end),
    "left"
)

in_gene = variant_genes.filter(F.col("gene_name").isNotNull()).count()
total = variants.count()

print(f"Variants in genes: {in_gene:,}")
print(f"Intergenic variants: {total - in_gene:,}")
print(f"Percentage in genes: {in_gene/total*100:.1f}%")
print(f"Percentage intergenic: {(total - in_gene)/total*100:.1f}%")

In [ ]:
# Top 10 genes by variant count
top_genes = variant_genes \
    .filter(F.col("gene_name").isNotNull()) \
    .groupBy("gene_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10)

top_genes.show()

# ANSWER: The join DOES cause a shuffle (SortMergeJoin by default).
# Alternative: broadcast join avoids the shuffle entirely.
print("\nAlternative: Broadcast join")
variant_genes_bc = variants.join(
    F.broadcast(genes),
    (variants.CHROM == genes.chrom) &
    (variants.POS >= genes.start) &
    (variants.POS <= genes.end),
    "left"
)
variant_genes_bc.explain()

---
## Task 6: Performance Comparison - SOLUTION

In [ ]:
# Spark (cold)
start = time.time()
spark_result = variants.groupBy("CHROM").agg(
    F.count("*").alias("count"),
    F.mean("QUAL").alias("mean_qual")
).collect()
spark_time = time.time() - start
print(f"Spark (cold): {spark_time:.3f}s")

# pandas
start = time.time()
pandas_df = pd.read_csv(os.path.join(data_dir, "variants_medium.csv"))
pandas_result = pandas_df.groupby("CHROM").agg(
    count=("CHROM", "size"),
    mean_qual=("QUAL", "mean")
).reset_index()
pandas_time = time.time() - start
print(f"Pandas:       {pandas_time:.3f}s")

In [ ]:
# Cached Spark
variants.cache()
variants.count()  # trigger cache

start = time.time()
spark_cached_result = variants.groupBy("CHROM").agg(
    F.count("*").alias("count"),
    F.mean("QUAL").alias("mean_qual")
).collect()
spark_cached_time = time.time() - start

variants.unpersist()

print(f"Pandas:         {pandas_time:.3f}s")
print(f"Spark (cold):   {spark_time:.3f}s")
print(f"Spark (cached): {spark_cached_time:.3f}s")

# ANSWERS:
# - On 100K rows, pandas is typically faster due to Spark overhead
# - Spark becomes necessary when data exceeds memory (>1-10 GB)
# - Spark overhead: JVM startup, query planning, serialization,
#   shuffle I/O, task scheduling

---
## Task 7: Optimization Challenge - SOLUTION

In [ ]:
# UNOPTIMIZED
genes_for_join = spark.read.csv(
    os.path.join(data_dir, "genes.bed"),
    sep="\t", header=False,
    schema="chrom STRING, start INT, end INT, gene_name STRING, strand STRING"
)

start = time.time()
result_unoptimized = variants \
    .join(genes_for_join,
          (variants.CHROM == genes_for_join.chrom) &
          (variants.POS >= genes_for_join.start) &
          (variants.POS <= genes_for_join.end)) \
    .filter(F.col("AF") < 0.01) \
    .select(variants.CHROM, "POS", "gene_name", "AF")
count_unopt = result_unoptimized.count()
time_unoptimized = time.time() - start
print(f"Unoptimized: {count_unopt} variants in {time_unoptimized:.3f}s")
print("\nUnoptimized plan:")
result_unoptimized.explain()

In [ ]:
# OPTIMIZED: Three key optimizations
# 1. Pre-filter variants (reduces data shuffled)
# 2. Broadcast the genes table (avoids shuffle)
# 3. Select only needed columns early

start = time.time()

rare_variants = variants.filter(F.col("AF") < 0.01) \
    .select("CHROM", "POS", "AF")

result_optimized = rare_variants.join(
    F.broadcast(genes_for_join),
    (rare_variants.CHROM == genes_for_join.chrom) &
    (rare_variants.POS >= genes_for_join.start) &
    (rare_variants.POS <= genes_for_join.end)
).select(rare_variants.CHROM, "POS", "gene_name", "AF")

count_opt = result_optimized.count()
time_optimized = time.time() - start

print(f"Optimized: {count_opt} variants in {time_optimized:.3f}s")
print(f"Speedup: {time_unoptimized/time_optimized:.1f}x")
print("\nOptimized plan:")
result_optimized.explain()

# ANSWER: Filtering before join helps the most. Broadcasting eliminates
# the shuffle. BroadcastHashJoin sends the small table to all workers,
# while SortMergeJoin requires sorting and shuffling both tables.

---
## Task 8: Write Results - SOLUTION

In [ ]:
output_path = os.path.join(data_dir, "filtered_variants.parquet")
filtered_variants.write.mode("overwrite").parquet(output_path)
print(f"Saved to {output_path}")

In [ ]:
# File size comparison
csv_path = os.path.join(data_dir, "variants_medium.csv")
csv_size = os.path.getsize(csv_path)

parquet_size = sum(
    os.path.getsize(os.path.join(dirpath, f))
    for dirpath, _, filenames in os.walk(output_path)
    for f in filenames
)

print(f"CSV size:     {csv_size / 1024 / 1024:.1f} MB")
print(f"Parquet size: {parquet_size / 1024 / 1024:.1f} MB")
print(f"Compression:  {csv_size / parquet_size:.1f}x smaller")

# ANSWER: Parquet is preferred because:
# 1. Columnar format - only reads columns needed
# 2. Built-in compression
# 3. Schema embedded
# 4. Predicate pushdown
# 5. Splittable for parallel reads

---
## Reflection - Model Answers

**1. When would you use Spark instead of pandas?**
When data exceeds available memory (typically >1-10 GB), when you need distributed
processing across a cluster, for complex ETL pipelines, or when fault tolerance matters.

**2. Main costs of distributed computing?**
Serialization overhead, network I/O for shuffles, JVM startup time, query planning,
operational complexity, debugging difficulty, and the learning curve.

**3. Designing a Spark job for 100K whole genomes?**
Partition data by chromosome, use Parquet format, broadcast small reference tables,
cache frequently accessed DataFrames, use columnar operations (avoid UDFs),
deploy on a cluster with sufficient memory per executor.

**4. Lessons from Spark UI?**
DAG visualization shows how operations are grouped into stages. Shuffle read/write
sizes reveal data movement costs. Task duration distributions show data skew.
The SQL tab shows how the Catalyst optimizer rewrites queries.

In [ ]:
spark.stop()
print("Solutions notebook complete!")